# Feature engineering
In this step we will introduce features learned from the data exploration step, re-formatting the information we have into a language the model can understand. <br>
This includes: <br>
- splitting wind speed into discrete bins to approach the "S" curve <br>
- splitting wind directions into discrete bins (NE, NW, SW, SE) <br>
- create cycling time feature (hour_of_day, month_of_year)


## Import necessary libraries

In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, r2_score, mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint, uniform


## Load data

In [54]:
path1 = "../Data/Processed/wind_persistance_polar.csv"
path2 = "../Data/Processed/wind_average_polar.csv"
path3 = "../Data/Processed/wind_hourly_polar.csv"

def load_data(path):
    data = pd.read_csv(path)
    return data

wind_persistance_polar = load_data(path1)
wind_average_polar = load_data(path2)
wind_hourly_polar = load_data(path3)

wind_persistance_polar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27048 entries, 0 to 27047
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   TIMESTAMP            27048 non-null  object 
 1   POWER                27048 non-null  float64
 2   wind_speed_10m       27048 non-null  float64
 3   wind_direction_10m   27048 non-null  float64
 4   wind_speed_100m      27048 non-null  float64
 5   wind_direction_100m  27048 non-null  float64
dtypes: float64(5), object(1)
memory usage: 1.2+ MB


Somehow the timestamp is not datetime type 

## Create features

In [55]:
def create_features(wind):
    """
    Create features to improve wind power forecasting.
    """
    wind_feat = wind.copy()
    wind_feat['TIMESTAMP'] = pd.to_datetime(wind_feat['TIMESTAMP']) 

    # 1. Wind speed bins using 100m (linearizing S-curve)
    # Based on the plots, we define three regions:
    # a. Cut-in: 0-4 m/s
    # b. Ramp-up: 4-20 m/s
    # c. Cut-out: >20 m/s

    speed_bins = [0, 4, 20, 50]
    speed_labels = ['cut-in', 'ramp-up', 'cut-out']

    wind_feat['wind_speed_bin'] = pd.cut(wind_feat['wind_speed_100m'], bins=speed_bins, labels=speed_labels, right=False, include_lowest=True)

    # 2. Wind direction bins
    dir_bins = [0, 90, 180, 270, 360]
    dir_labels = ['NE', 'SE', 'SW', 'NW']
    wind_feat['wind_dir_bin'] = pd.cut(wind_feat['wind_direction_100m'], bins=dir_bins, labels=dir_labels, right=False, include_lowest=True)

    # 3. Cyclical Time Features (hour)
    wind_feat['hour'] = wind_feat['TIMESTAMP'].dt.hour
    wind_feat['hour_sin'] = np.sin(2 * np.pi * wind_feat['hour'] / 24)
    wind_feat['hour_cos'] = np.cos(2 * np.pi * wind_feat['hour'] / 24)

    # 4. Cyclical Time Features (month)
    wind_feat['month'] = wind_feat['TIMESTAMP'].dt.month
    wind_feat['month_sin'] = np.sin(2 * np.pi * wind_feat['month'] / 12)
    wind_feat['month_cos'] = np.cos(2 * np.pi * wind_feat['month'] / 12)

    return wind_feat


### Apply features to all datasets

In [56]:
# List of features to use
features_to_use = [
    'POWER',
    'wind_speed_100m',
    'wind_speed_bin',
    'wind_dir_bin',
    'hour_sin',
    'hour_cos',
    'month_sin',
    'month_cos'
]

# Create features for persistence data
wind_pers_feat = create_features(wind_persistance_polar)
wind_pers_feat = wind_pers_feat[features_to_use].copy()

# Create features for average data
wind_avg_feat = create_features(wind_average_polar)
wind_avg_feat = wind_avg_feat[features_to_use].copy()

# Create features for hourly data
wind_hour_feat = create_features(wind_hourly_polar)
wind_hour_feat = wind_hour_feat[features_to_use].copy()

print("\nDataFrames are ready for preprocessing.")


DataFrames are ready for preprocessing.


## Create preprocessing pipeline

In [57]:
def get_preprocessor():
    """
    Define and return preprocessing pipeline.
    """
    # 1. Define feature lists
    numerical_features = ['wind_speed_100m']
    categorical_features = ['wind_speed_bin', 'wind_dir_bin']
    passthrough_features = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos']

    # 2. Define the transformers
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    # 3. Create the ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical_features),
            ('cat', categorical_transformer, categorical_features),
            ('pass', 'passthrough', passthrough_features)
        ]
    )

    return preprocessor


### Apply preprocessing to all datasets

Split feature data into X (input data) and y (output data). <br>
Fit the preprocessing pipeline to all three datasets.

In [58]:
preprocessor = get_preprocessor()

# Apply preprocessing to persistance data
# 1. Split into X and y
X_pers = wind_pers_feat.drop('POWER', axis=1)
y_pers_train = wind_pers_feat['POWER']

# 2. Fit preprocessing on training data
X_pers_train = preprocessor.fit_transform(X_pers)
print("\nTransformed training data shape (persistence data):", X_pers_train.shape)

# Apply preprocessing for average data
# 1. Split into X and y
X_avg = wind_avg_feat.drop('POWER', axis=1)
y_avg_train = wind_avg_feat['POWER']

# 2. Fit preprocessing on training data
X_avg_train = preprocessor.fit_transform(X_avg)
print("\nTransformed training data shape (average data):", X_avg_train.shape)

# Apply preprocessing for hourly data
X_hour = wind_hour_feat.drop('POWER', axis=1)
y_hour_train = wind_hour_feat['POWER']

# 2. Fit preprocessing on training data
X_hour_train = preprocessor.fit_transform(X_hour)
print("\nTransformed training data shape (hourly data):", X_hour_train.shape)


Transformed training data shape (persistence data): (27048, 12)

Transformed training data shape (average data): (27048, 12)

Transformed training data shape (hourly data): (27048, 12)


### Machine learning model evaluation

As the wind speed and power output relationship is non-linear, we will compare the results of a Gradient Boosting model (e.g., XGBoost) with a Random Forest model. These will also be compared to a Linear Regression to further exemplify the non-linearity of the data set. <br>
We will evaluate the models performance using cross-validation and their results will be compared using MSE (Mean Square Error) and R² (Coefficient of Determination). <br>
The highest scoring model will be chosen for further fine tuning.

In [59]:
# XGBoost with tuned parameters for wind power forecasting
xgb_model = XGBRegressor(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror',
    verbosity=0,
    n_jobs=-1
)


In [60]:
# Random Forest regressor
rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)


In [61]:
#  Linear Regression
linear_model = LinearRegression()


First we evaluate the three models on the Persistence Fill dataset.

In [62]:
#Results of all ML models for the Persistance Fill data set
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42)
}

scoring = {
    'r2': make_scorer(r2_score),
    'mae': make_scorer(mean_absolute_error, greater_is_better=False)
}

results = {}
for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    # Use the raw DataFrame X_pers so ColumnTransformer can select columns by name
    cv_results = cross_validate(pipeline, X_pers, y_pers_train, cv=5, scoring=scoring, n_jobs=-1)
    results[name] = cv_results

# Summarize results into a small DataFrame and print
summary_list = []
for name, res in results.items():
    mean_r2 = np.mean(res['test_r2'])
    mean_mae = np.mean(res['test_mae'])
    # If MAE was returned as negative (scoring with greater_is_better=False), take absolute
    if mean_mae < 0:
        mean_mae = -mean_mae
    summary_list.append({'model': name, 'mean_r2': mean_r2, 'mean_mae': mean_mae})

print("Results for Persistance Fill")
summary_df = pd.DataFrame(summary_list).set_index('model').sort_values('mean_r2', ascending=False)
print(summary_df)


Results for Persistance Fill
                    mean_r2  mean_mae
model                                
XGBoost            0.769485  0.107687
Random Forest      0.762075  0.109897
Linear Regression  0.752136  0.122348


In [63]:
X_pers.shape

(27048, 7)

In [64]:
X_pers_train.shape

(27048, 12)

Now we evaluate the three models on the Moving Average Fill dataset.

In [65]:
#Results of all ML models for the Moving Average Fill dataset
results = {}
for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    # Use the raw DataFrame X_pers so ColumnTransformer can select columns by name
    cv_results = cross_validate(pipeline, X_avg, y_avg_train, cv=5, scoring=scoring, n_jobs=-1)
    results[name] = cv_results

# Summarize results into a small DataFrame and print
summary_list = []
for name, res in results.items():
    mean_r2 = np.mean(res['test_r2'])
    mean_mae = np.mean(res['test_mae'])
    # If MAE was returned as negative (scoring with greater_is_better=False), take absolute
    if mean_mae < 0:
        mean_mae = -mean_mae
    summary_list.append({'model': name, 'mean_r2': mean_r2, 'mean_mae': mean_mae})
print("Results for Moving Average Fill")
summary_df = pd.DataFrame(summary_list).set_index('model').sort_values('mean_r2', ascending=False)
print(summary_df)

Results for Moving Average Fill
                    mean_r2  mean_mae
model                                
XGBoost            0.768333  0.108015
Random Forest      0.761863  0.109904
Linear Regression  0.752020  0.122362


Lastly we evaluate the three models on the Climatology Fill dataset.

In [66]:
#Results of all ML models for the Hourly Climatology Fill data set
results = {}
for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    # Use the raw DataFrame X_pers so ColumnTransformer can select columns by name
    cv_results = cross_validate(pipeline, X_hour, y_hour_train, cv=5, scoring=scoring, n_jobs=-1)
    results[name] = cv_results

# Summarize results into a small DataFrame and print
summary_list = []
for name, res in results.items():
    mean_r2 = np.mean(res['test_r2'])
    mean_mae = np.mean(res['test_mae'])
    # If MAE was returned as negative (scoring with greater_is_better=False), take absolute
    if mean_mae < 0:
        mean_mae = -mean_mae
    summary_list.append({'model': name, 'mean_r2': mean_r2, 'mean_mae': mean_mae})
print("Results for Hourly Climatology Fill")
summary_df = pd.DataFrame(summary_list).set_index('model').sort_values('mean_r2', ascending=False)
print(summary_df)

Results for Hourly Climatology Fill
                    mean_r2  mean_mae
model                                
XGBoost            0.768326  0.108118
Random Forest      0.760862  0.110297
Linear Regression  0.751191  0.122494


After cross-validation all models had quite high R2 and low MAE scores. However, the XGBoost model performed the best in both metrics, thus this model is chosen for further fine tuning. Additionally, the R2 score was the highest and MAE the lowest for the Persistence Fill dataset, so this method of filling in the missing power output values is chosen as the best option.

Comment: Experiment with the rolling average and further linearisation of the S-curve some more

### Fine Tuning
The best performing model (XGBoost) will be fine tuned using RandomizedSearchCV to find the optimal hyperparameters to maximize R2 score. <br>

In [67]:
# Fine-tune XGBoost hyperparameters with RandomizedSearchCV (maximize R2)
from sklearn.pipeline import Pipeline as SKPipeline

# Create a fresh XGBoost estimator inside a pipeline so params can be prefixed with 'model__'
xgb_base = XGBRegressor(objective='reg:squarederror', verbosity=0, n_jobs=-1, random_state=42)
pipeline_rs = SKPipeline([('preprocessor', preprocessor), ('model', xgb_base)])

# Parameter distributions (prefixed with 'model__' to target the estimator in the pipeline)
param_dist = {
    'model__n_estimators': randint(100, 1200),
    'model__max_depth': randint(3, 12),
    'model__learning_rate': uniform(0.01, 0.29),
    'model__subsample': uniform(loc=0.5, scale=0.5),
    'model__colsample_bytree': uniform(loc=0.5, scale=0.5),
    'model__gamma': uniform(loc=0, scale=5),
    'model__min_child_weight': randint(1, 10)
}

# TimeSeries CV to avoid leakage (use the same split count used earlier)
tscv_tune = TimeSeriesSplit(n_splits=5)

# Randomized search (optimize for R2). n_iter set modestly to keep runtime reasonable; increase if you have time.
rs = RandomizedSearchCV(
    estimator=pipeline_rs,
    param_distributions=param_dist,
    n_iter=40,
    scoring='r2',
    cv=tscv_tune,
    random_state=42,
    n_jobs=-1,
    verbose=2,
    return_train_score=False
)

# Fit the search on the Persistence Fill dataset
# This step takes quite a lot of time depending on n_iter and dataset size. May need optimization.

rs.fit(X_pers, y_pers_train)

# Save best estimator and results for later reuse
best_xgb = rs.best_estimator_
best_params = rs.best_params_
best_score = rs.best_score_
print('Best R2 (cv):', best_score)
print('Best params:')
for k, v in best_params.items():
    print(f'  {k}: {v}')



Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best R2 (cv): 0.779441023103767
Best params:
  model__colsample_bytree: 0.7850305850446825
  model__gamma: 0.4858824688538427
  model__learning_rate: 0.1883520957427592
  model__max_depth: 8
  model__min_child_weight: 7
  model__n_estimators: 1061
  model__subsample: 0.6270818245348694


After fine tuning the XGBoost model using RandomizedSearchCV, the best parameters were found to be:
- n_estimators: 1061
- max_depth: 8
- learning_rate: 0.1884
- subsample: 0.6271
- colsample_bytree: 0.7850
- gamma: 0.4859
- min_child_weight: 7 <br>
The R2 score achieved with these parameters was 0.779, which is a small, but significant improvement from the previously chosen parameters.

In [68]:
# Summarize top 5 results
cv_results = pd.DataFrame(rs.cv_results_)
top = cv_results.sort_values('mean_test_score', ascending=False).head(5)[[
    'mean_test_score', 'std_test_score', 'params'
]]
print(top)



    mean_test_score  std_test_score  \
36         0.779441        0.034599   
10         0.779383        0.036241   
20         0.778069        0.033731   
15         0.777027        0.035031   
25         0.776815        0.033879   

                                               params  
36  {'model__colsample_bytree': 0.7850305850446825...  
10  {'model__colsample_bytree': 0.5993578407670862...  
20  {'model__colsample_bytree': 0.7513395116144308...  
15  {'model__colsample_bytree': 0.8477580432130638...  
25  {'model__colsample_bytree': 0.6705331755251293...  


### Preprocessing test data and evaluating fine-tuned model

Before evaluating the fine-tuned model on the test set, the test data must first be preprocessed using the same preprocessing pipeline as the training data. <br>
The fine-tuned model is then used to make predictions on the preprocessed test data. The performance of the model is evaluated using R².

In [69]:
# Import processed weather data for test set evaluation
path4 = "../Data/Processed/weather_input_polar.csv"
# Importing the solution weather data to evaluate the fine-tuned model on the test set
path5 = "../Data/Solution_A.csv"

# Importing the Forecast template_A to create the submission file later
#path6 = "../Data/Processed/ForecastTemplate_A.csv"

def load_data(path):
    data = pd.read_csv(path)
    
    return data

weather_input_polar = load_data(path4)
solution_A_power = load_data(path5)

print(weather_input_polar.info())
print(weather_input_polar.head())
print("\n")
print(solution_A_power.info())
print(solution_A_power.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 672 entries, 0 to 671
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   TIMESTAMP            672 non-null    object 
 1   wind_speed_10m       672 non-null    float64
 2   wind_direction_10m   672 non-null    float64
 3   wind_speed_100m      672 non-null    float64
 4   wind_direction_100m  672 non-null    float64
dtypes: float64(4), object(1)
memory usage: 26.4+ KB
None
             TIMESTAMP  wind_speed_10m  wind_direction_10m  wind_speed_100m  \
0  2018-02-01 01:00:00       10.425294           22.821322        13.633243   
1  2018-02-01 02:00:00       10.291520           21.521575        13.714049   
2  2018-02-01 03:00:00        9.831939           22.696497        12.991587   
3  2018-02-01 04:00:00        9.369325           30.922131        12.659667   
4  2018-02-01 05:00:00        9.613713           32.270600        12.933785   

   wind_direc

The Weather Input data does not have the same features as the training data, we need to ensure that the preprocessing pipeline can handle this discrepancy. Namely the POWER column is missing and the categorical features such as wind direction bins are not present. <br>

Also the timestamp of the Solution_A_power needs to be converted to datetime type. <br>

Comment: I'm not sure this is a good way of actually implementing weather input data for the final prediction. As the weather input data doesn't contain the 'POWER' column, which is expected by the preprocessor, I just added the missing column from the Solution_A.csv.

In [70]:
# Create a new DataFrame `weather_input_polar_power` that includes the
# POWER column from `solution_A_power` placed immediately after TIMESTAMP.

# Work on a copy so we don't modify the original inputs in-place
weather_input_polar_power = weather_input_polar.copy()

# Ensure TIMESTAMP columns are comparable (convert in copies only)
weather_input_polar_power['TIMESTAMP'] = pd.to_datetime(weather_input_polar_power['TIMESTAMP'])
solution_power_for_merge = solution_A_power.copy()
solution_power_for_merge['TIMESTAMP'] = pd.to_datetime(solution_power_for_merge['TIMESTAMP'], format='%Y%m%d %H:%M', errors='coerce')

# Merge POWER into the weather data (left join to preserve weather rows).
weather_input_polar_power = weather_input_polar_power.merge(
    solution_power_for_merge[['TIMESTAMP', 'POWER']],
    on='TIMESTAMP',
    how='left'
)

# Reorder columns: place 'POWER' immediately after 'TIMESTAMP'
cols = weather_input_polar_power.columns.tolist()
if 'POWER' in cols:
    cols.remove('POWER')
    insert_idx = cols.index('TIMESTAMP') + 1 if 'TIMESTAMP' in cols else 1
    cols.insert(insert_idx, 'POWER')
    weather_input_polar_power = weather_input_polar_power[cols]

# Check the result
weather_input_polar_power.head()


,TIMESTAMP,POWER,wind_speed_10m,wind_direction_10m,wind_speed_100m,wind_direction_100m
0,2018-02-01 01:00:00,0.876351,10.425294,22.821322,13.633243,21.674865
1,2018-02-01 02:00:00,0.921515,10.291520,21.521575,13.714049,20.695572
2,2018-02-01 03:00:00,0.895380,9.831939,22.696497,12.991587,21.447003
3,2018-02-01 04:00:00,0.898016,9.369325,30.922131,12.659667,29.063365
4,2018-02-01 05:00:00,0.862974,9.613713,32.270600,12.933785,30.554298


In [71]:
# Prepare and run the weather input through the same preprocessing pipeline used for training
# 1) Create the same engineered features on the weather input DataFrame
weather_input_feat = create_features(weather_input_polar_power)
# Keep the same columns used for training (features_to_use defined earlier)
weather_input_feat = weather_input_feat[features_to_use].copy()

# 2) Split into X and y for the weather input (POWER may be present from the merge)
X_weather_input = weather_input_feat.drop('POWER', axis=1) if 'POWER' in weather_input_feat.columns else weather_input_feat.copy()
y_weather_input = weather_input_feat['POWER'] if 'POWER' in weather_input_feat.columns else None

# 3) Fit the preprocessor on the persistence training data (fit once), then transform both sets
# This is a check, so I can be sure the preprocessor results are consistent with earlier steps
# Can delete this step later
preprocessor = get_preprocessor()
preprocessor.fit(X_pers)  # X_pers was created earlier from wind_pers_feat
X_pers_train = preprocessor.transform(X_pers)
print("\nTransformed training data shape (persistence data):", X_pers_train.shape)

# 4) Transform weather input using the fitted preprocessor
X_weather_input_transformed = preprocessor.transform(X_weather_input)
print("\nTransformed weather input data shape (weather input):", X_weather_input_transformed.shape)


Transformed training data shape (persistence data): (27048, 12)

Transformed weather input data shape (weather input): (672, 12)


### Evaluation of fine-tuned model on test set

In [72]:
#Evaluate the fine tuned XGBoost model on the preprocessed weather input data set using R2 score
weather_input_test = rs.best_estimator_

X_weather_input = weather_input_feat.drop('POWER', axis=1)
y_weather_input = weather_input_feat['POWER'].copy()

# Predict on the power data
X_weather_input_transformed = preprocessor.transform(X_weather_input)
final_predictions = weather_input_test.predict(X_weather_input)

final_r2 = r2_score(y_weather_input, final_predictions)
final_mae = mean_absolute_error(y_weather_input, final_predictions)
print("Final R2 on Weather_Forecast_Input_A test set:", final_r2)
print("Final MAE on Weather_Forecast_Input_A test set:", final_mae)


Final R2 on Weather_Forecast_Input_A test set: 0.8629773633082394
Final MAE on Weather_Forecast_Input_A test set: 0.09428445967330851


### Comments:
 It's a bit unintuitive that the previously preprocessed weather input data is being preprocessed again here, but the previous check is useful to ensure that the preprocessing pipeline can handle the missing features in the weather input data.

 The final R2 score for the test dataset (Weather_Forecast_Input_A) is 0.8630 and MAE is 0.0943. Personally I think these results are suspiciously good, since the cross-validated R2 score was only 0.779. Please double check that everything is correct here.

Perhaps the results could be improved by some feature engineering, such as increasing the importance of the power forecast in training specifically for February, since that is the month for which the model is being tested.

TODOs: 
Convert the final model evaluation from this notebook in a way that can easily be applied to the 4_forecast_evaluation notebook.



In [73]:
X_weather_input.shape

(672, 7)

In [74]:
X_weather_input_transformed.shape

(672, 12)